# 07 - Cloud readiness

Check Docker and Azure CLI before deployment. This notebook only reads local tool and login state. It does not create resources or print account IDs or tokens.

In [1]:
from pathlib import Path
import json
import shutil
import subprocess

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if p.name == "MLOP-PJ" and (p / "README.md").is_file())
state = {"docker_cli": shutil.which("docker") is not None,
         "azure_cli": shutil.which("az") is not None,
         "resources_created": False}
if state["docker_cli"]:
    check = subprocess.run(["docker", "version", "--format", "{{.Server.Version}}"],
                           capture_output=True, text=True, timeout=20)
    state["docker_ready"] = check.returncode == 0
    state["docker_server_version"] = check.stdout.strip() if check.returncode == 0 else None
else:
    state["docker_ready"] = False
if state["azure_cli"]:
    check = subprocess.run(["az", "account", "show", "--query", "{state:state}", "-o", "json"],
                           capture_output=True, text=True, timeout=30)
    state["azure_logged_in"] = check.returncode == 0
    state["subscription_enabled"] = check.returncode == 0 and json.loads(check.stdout).get("state") == "Enabled"
else:
    state["azure_logged_in"] = False
    state["subscription_enabled"] = False
(ROOT / "reports/cloud-readiness.json").write_text(json.dumps(state, indent=2) + "\n")
print(json.dumps(state, indent=2))


{
  "docker_cli": true,
  "azure_cli": true,
  "resources_created": false,
  "docker_ready": true,
  "docker_server_version": "29.4.0",
  "azure_logged_in": true,
  "subscription_enabled": true
}


## Checkpoint

Send the readiness result after 05 and 06 pass. A false value means setup is still needed, not that the model failed. Do not create cloud resources yet. Region, quota, permissions, gross cost, and the 150 THB proposal budget still need review. Keep credentials out of notebook output.